# 05a — A neural network from scratch — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/05.md) · [Course map](../PLAN.md) · [Project](../../projects/stage05/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Implement forward and backward passes
- Track matrix dimensions
- Verify gradients before training


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: A neural network as connected arithmetic

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

Start with the derivative and chain-rule explanations in Stage 02. A neuron computes a weighted sum plus bias, then applies an activation function. A layer groups neurons; a network composes layers. Without nonlinear activations, multiple linear layers still represent a linear transformation.

For a batch X with shape `(examples, features)`, multiply by W with shape `(features, hidden_units)` to obtain `(examples, hidden_units)`. Add one bias per hidden unit. tanh compresses values to (-1,1), with derivative `1-tanh(z)**2`.

Forward propagation calculates predictions and loss. Backpropagation applies the chain rule in reverse to calculate how each parameter affects loss. An optimizer uses those gradients to update parameters. SGD follows the current gradient; momentum also carries a decaying history of earlier gradients.

### Worked example: follow the values

XOR outputs 1 when exactly one of two inputs is 1. Its four input/target pairs cannot be separated by one straight line, making it a small nonlinear example. Check gradients before training: perturb one weight by plus/minus h, compute both losses, divide their difference by 2h, and compare to backpropagation.

### Pause and explain

Why restore a weight after a finite-difference check?

<details><summary>Check your reasoning after trying</summary>

Otherwise later checks and training use a different parameter state, so the comparisons no longer refer to the same network.

</details>

### Common mistakes to check

Overfitting four XOR examples tests capacity and optimization, not generalization. Never update a weight before computing all gradients that depend on its old value.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

A neuron computes a weighted sum plus bias, followed by an activation. Layers compose these transformations. Without nonlinear activations, stacked linear layers collapse into one linear transformation. ReLU keeps positive values, sigmoid maps to (0,1), tanh to (-1,1), and softmax normalizes class scores.

For a batch X of shape (n,d), a weight matrix W of shape (d,h) gives hidden activations (n,h). A loss measures prediction error. Backpropagation applies the chain rule to compute parameter gradients. The optimizer updates parameters; it does not compute the forward loss itself.

MSE is common for regression. Binary cross entropy suits a binary target; multiclass cross entropy uses one target class per example. Stable implementations operate on logits to avoid logarithms of rounded probabilities. Verify individual gradients numerically before trusting a loss curve. Here a one-layer regression network isolates the mechanics; the project extends it to two layers and XOR.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
import numpy as np
x = np.array([[1., 2.], [3., 4.]])
w = np.array([[.2], [.3]])
print((x @ w).shape)


## Exercise 1: Stable sigmoid

Return a sigmoid for NumPy input without overflowing exp on large magnitudes.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def sigmoid(x):
    import numpy as np
    x = np.asarray(x, dtype=float)
    z = np.exp(-np.abs(x))
    return np.where(x >= 0, 1/(1+z), z/(1+z))


In [ ]:
assert np.allclose(sigmoid([-1000, 0, 1000]), [0, .5, 1])
print("Exercise 1: checks passed")


**Why this works:** Numerical stability is part of correctness, not a cosmetic optimization.


## Exercise 2: Linear backward pass

For X(n,d), y(n,1), w(d,1), scalar b, return MSE, dw, db.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def backward(x, y, w, b):
    residual = x @ w + b - y
    loss = float((residual**2).mean())
    return loss, 2*x.T@residual/len(x), float(2*residual.mean())


In [ ]:
x = np.array([[1.], [2.]])
y = 2*x
w = np.zeros((1, 1))
loss, dw, db = backward(x, y, w, 0)
assert loss == 10 and np.allclose(dw, [[-10]]) and db == -6
print("Exercise 2: checks passed")


**Why this works:** Batch averaging must be applied consistently to loss and gradients.


## Exercise 3: Training loop

Use backward to fit w,b from zeros with 500 updates and lr=.05. Return w,b,loss history.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def train_linear(x, y, steps=500, lr=.05):
    import numpy as np
    w, b, history = np.zeros((x.shape[1], 1)), 0., []
    for _ in range(steps):
        loss, dw, db = backward(x, y, w, b)
        history.append(loss)
        w -= lr*dw
        b -= lr*db
    return w, b, history


In [ ]:
w, b, history = train_linear(np.array([[-1.], [0.], [1.]]), np.array([[-1.], [1.], [3.]]))
assert history[-1] < 1e-8 and np.allclose(w, [[2]], atol=1e-3)
print("Exercise 3: checks passed")


**Why this works:** Mixing updated and old parameters within one backward pass produces incorrect gradients.


## Independent transfer

Implement a NumPy two-layer tanh network for XOR. Check 10 random gradient entries, overfit four examples, then compare SGD and momentum. Plot loss versus epoch.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [05a interface and acceptance cases](../assignments/05a.md).
2. Copy the [blank starter](../assignments/starters/05a.py) to `work/assignments/05a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 05a --solution work/assignments/05a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/05a.md#05a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 05a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Why an activation?

   It enables nonlinear functions; otherwise multiple affine layers remain affine.

2. What is backpropagation?

   Efficient reverse accumulation of derivatives through a computation graph.


## Reading and review

- [Primary reference 1](https://d2l.ai/chapter_multilayer-perceptrons/backprop.html)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
